In [ ]:
# -*- coding: utf-8 -*-

# CIFAR-10 — Baseline vs. Auto-Filtered vs. Human-in-the-Loop

This notebook trains and compares three versions of the same multitask (classification + reconstruction) CNN on **CIFAR-10**, using DINO-based foreground/background segmentation as a preprocessing step:

1. **Baseline** — trained with standard on-the-fly augmentation.
2. **Human-in-the-loop** — trained with a mix of standard augmentation plus augmented candidates a human manually approved/rejected via an in-notebook review widget.
3. **Auto-filtered** — trained with augmented candidates auto-approved by the *baseline* model's own confidence score (no human review), as a cheaper alternative to (2).

All three are evaluated on **clean accuracy** and **adversarial robustness** (FGSM and PGD attacks), and the pipeline is then repeated with transfer-learning backbones (MobileNetV2, ResNet50, VGG16, VGG19).

**How to run:** top-to-bottom for a full fresh run, or use the shortcut cells to load cached segmentation/candidates/models from Drive and skip the slow steps. The human review step (section 5) is interactive.

> **Cleanup note:** a few small things were changed from the raw export, all logic-preserving:
> - Cell 44: dedented 3 lines that had a dangling indent (IndentationError if run standalone) -- same pattern as the STL-10 notebook. No logic change.
> - Cell 56: removed a two-way (baseline vs. human) FGSM results table print -- superseded by the three-way (baseline/auto/human) print further down. The FGSM accuracy computation itself is unchanged and still runs (those numbers are reused in the three-way comparison).
> - Cell 57: removed a two-way (baseline vs. human) PGD results table print plus a matching plot saved as adversarial_comparison_seg.png -- both fully superseded by the three-way (baseline/auto/human) version further down the notebook (three_way_robustness.png), which covers the same two models plus the auto-filtered one. The PGD accuracy computation itself is unchanged and still runs (those numbers are reused in the three-way comparison).
> - Cell 69: corrected a stale comment ('freeze everything except the last 20 layers') that didn't match the actual code (`layers[:-35]`, i.e. last 35). Comment-only fix, no logic change.
> - Cell 77: removed a leftover print block that reported baseline_pgd_accs/human_pgd_accs (the Part 4 custom-model results) mislabeled inside the MobileNetV2 transfer section -- a copy-paste leftover printing unrelated numbers with no distinguishing label. Same pattern found and removed in the STL-10 notebook. The actual transfer-model FGSM numbers and the adversarial plot are unaffected.
> - A handful of large cells that bundled several unrelated functions together (e.g. augmentation + segmentation + model-building + training helpers all in one cell) were split into one cell per function for readability. No code was reordered or altered.
> - The interactive review widget's button labels (checkmark/cross/etc.) use a few symbol characters in the original source -- those are left exactly as they were, since they're functional UI text, not decorative additions. No emoji were added anywhere by this cleanup.
> - Everything else -- every model, every hyperparameter, every training call -- is unchanged.


In [ ]:
!pip install tensorflow==2.20.0 -q

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os, json, sys

In [ ]:
import torch
import torch.nn.functional as F
from PIL import Image
from scipy.ndimage import gaussian_filter

In [ ]:
import tensorflow as tf
import keras
from keras import layers, Model, Input
from keras.datasets import cifar10
from keras.optimizers import Adam

In [ ]:
# fix the random seed everywhere so runs are reproducible
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [ ]:
print(f'TensorFlow {tf.__version__}  |  Keras {keras.__version__}  |  NumPy {np.__version__}')

In [ ]:
# DINO (self-supervised Vision Transformer) is only used for its attention
# maps -- we never train it, just use it to roughly localize the foreground
# object in each image (see segment_batch() below).
print("loading DINO segmentation model...")
dino_model = torch.hub.load('facebookresearch/dino:main', 'dino_vits8')
dino_model.eval()
dino_model = dino_model.cuda()
print("DINO model ready")

In [ ]:
# ── model architecture ───────────────────────────────────────────────
LATENT_DIM = 128
BETA       = 0.5        # reconstruction loss weight (eq. 2 in proposal)

In [ ]:
# ── training ───────────────────────────────────────────────────────
BATCH_SIZE      = 64
M_VIEWS         = 3     # augmented copies per original (baseline generator)
EPOCHS_BASELINE = 40    # epochs for Part 1
EPOCHS_HUMAN    = 40    # epochs for Part 2  (match baseline for fair comparison)
STEPS_PER_EPOCH = 300   # ~300 steps x 64 batch = ~19k images/epoch
VAL_STEPS       = 50
LR              = 1e-3

In [ ]:
def make_lr_schedule(epochs, warmup=5):
    # linear warmup for the first `warmup` epochs, then cosine decay to 0
    # over the rest of training -- keeps early gradients from blowing up
    # the batchnorm stats while LR is still large.
    def schedule(epoch, lr):
        if epoch < warmup:
            return LR * (epoch + 1) / warmup
        progress = (epoch - warmup) / (epochs - warmup)
        return LR * 0.5 * (1 + np.cos(np.pi * progress))
    return tf.keras.callbacks.LearningRateScheduler(schedule, verbose=0)

# ── candidate generation (Part 2 Step 1) ──────────────────────────
N_ORIGINALS = 3000        # source images to draw augmented candidates from

In [ ]:
# ── paths ──────────────────────────────────────────────────────────
OUT_DIR       = os.getcwd()
CAND_DIR      = os.path.join(OUT_DIR, 'candidates')
CAND_PATH     = os.path.join(CAND_DIR, 'candidates.npz')       # unreviewed augmented candidates
STATE_PATH    = os.path.join(CAND_DIR, 'review_state.json')    # human review progress (resumable)
APPROVED_PATH = os.path.join(CAND_DIR, 'approved_seg.npz')     # human-approved candidates

In [ ]:
# accumulates results for the comparison in Part 3
results = {}

In [ ]:
print(f'Output directory: {OUT_DIR}')

In [ ]:
# ── data loading ───────────────────────────────────────────────────

In [ ]:
# modified loading function for CIFAR-10 dataset
def load_cifar():
    # convenience loader using Keras' built-in CIFAR-10 fetcher (not used
    # in this run -- see load_cifar_from_drive() below, which loads from a
    # local tar.gz instead)
    print("loading cifar...")
    (x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()
    x_train = x_train.reshape(-1, 32, 32, 3).astype('float32') / 255.0
    x_test  = x_test.reshape(-1, 32, 32, 3).astype('float32') / 255.0
    y_train = y_train.flatten().astype('int32')
    y_test  = y_test.flatten().astype('int32')
    print(f'train: {x_train.shape}   test: {x_test.shape}')
    return x_train, x_test, y_train, y_test

# ── vectorized preprocessing (operates on whole batches at once) ───
# These replace the per-image Python loops that made the old generator slow.

In [ ]:
def augment_cifar(imgs, rng):
    # Applies a stack of random augmentations to a whole batch at once.
    # `rng` is a numpy Generator passed in by the caller so augmentation
    # randomness stays reproducible across a run (seeded from SEED).
    n = len(imgs)
    out = imgs.copy()

    # random horizontal flip (50% of images)
    flip = rng.random(n) < 0.5
    out[flip] = out[flip, :, ::-1, :]

    # random rotation (+/-30 degrees), 50% of images
    from scipy.ndimage import rotate
    for i in np.where(rng.random(n) < 0.5)[0]:
        angle = rng.uniform(-30, 30)
        out[i] = np.clip(rotate(out[i], angle, reshape=False), 0, 1)

    # aggressive random crop -- crop to ~18-26px (of 32) then upscale back
    # to 32x32 (40% of images); simulates zoomed-in/cropped views
    for i in np.where(rng.random(n) < 0.4)[0]:
        crop_size = rng.integers(18, 26)
        y = rng.integers(0, 32 - crop_size)
        x = rng.integers(0, 32 - crop_size)
        cropped = out[i, y:y+crop_size, x:x+crop_size, :]
        out[i] = np.array(Image.fromarray(
            (cropped * 255).astype(np.uint8)).resize((32, 32))) / 255.0

    # brightness jitter -- scales every pixel of an image by the same
    # random factor in [0.75, 1.25]
    factors = 1.0 + rng.uniform(-0.25, 0.25, size=(n, 1, 1, 1))
    out = np.clip(out * factors, 0.0, 1.0)

    # color channel jitter -- scales R, G, B channels independently by
    # random factors in [0.8, 1.2], simulating white-balance shifts
    for c in range(3):
        factors_c = 1.0 + rng.uniform(-0.20, 0.20, size=(n, 1, 1))
        out[:, :, :, c] = np.clip(out[:, :, :, c] * factors_c, 0.0, 1.0)

    # grayscale conversion on some images (30%)
    for i in np.where(rng.random(n) < 0.3)[0]:
        gray = out[i].mean(axis=2, keepdims=True)
        out[i] = np.repeat(gray, 3, axis=2)

    # large random erasing -- blacks out a random rectangle (cutout-style),
    # on 50% of images
    for i in np.where(rng.random(n) < 0.5)[0]:
        eh = rng.integers(8, 16);  ew = rng.integers(8, 16)
        ey = rng.integers(0, 32 - eh);  ex = rng.integers(0, 32 - ew)
        out[i, ey:ey+eh, ex:ex+ew, :] = 0.0

    return out.astype('float32')

In [ ]:
def segment_batch(imgs):
    """Use DINO attention to identify foreground, then blur background
    instead of zeroing it out — preserves more visual info for training."""
    from scipy.ndimage import gaussian_filter

    # DINO expects 224x224 inputs, so upscale from CIFAR-10's native 32x32
    imgs_resized = np.stack([
        np.array(Image.fromarray(
            (img * 255).astype(np.uint8)).resize((224, 224))) / 255.0
        for img in imgs
    ])

    # standard ImageNet normalization, required by the pretrained DINO weights
    tensor = torch.tensor(imgs_resized).permute(0, 3, 1, 2).float().cuda()
    mean = torch.tensor([0.485, 0.456, 0.406]).cuda().view(1, 3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).cuda().view(1, 3, 1, 1)
    tensor = (tensor - mean) / std

    # pull the CLS token's attention over all patch tokens from the last
    # transformer block -- this is DINO's well-known emergent foreground map
    with torch.no_grad():
        attentions = dino_model.get_last_selfattention(tensor)

    nh = attentions.shape[1]
    attentions = attentions[:, :, 0, 1:].reshape(len(imgs), nh, 28, 28)
    attentions = attentions.mean(dim=1)  # average over attention heads

    # normalize each image's attention map to [0, 1] independently
    att_min = attentions.flatten(1).min(1).values.view(-1, 1, 1)
    att_max = attentions.flatten(1).max(1).values.view(-1, 1, 1)
    attentions = (attentions - att_min) / (att_max - att_min + 1e-8)

    # binarize: anything above 0.1 normalized attention counts as foreground
    mask = (attentions > 0.1).float()

    # upsample the 28x28 attention mask back to the original 32x32 resolution
    mask_np = F.interpolate(
        mask.unsqueeze(1).cpu(), size=(32, 32), mode='bilinear', align_corners=False
    ).squeeze(1).numpy()
    result = []
    for i, img in enumerate(imgs):
        m = mask_np[i]
        # blur the whole image
        blurred = np.stack([
            gaussian_filter(img[:, :, c], sigma=2)
            for c in range(3)], axis=-1)
        # blend: sharp where foreground, blurred where background
        mask_rgb = np.repeat(m[..., np.newaxis], 3, axis=-1)
        combined = img * mask_rgb + blurred * (1 - mask_rgb)
        result.append(combined.astype('float32'))

    return np.stack(result)

In [ ]:
def segment_all(imgs, chunk_size=200):
  # runs segment_batch() over the full array in chunks, since doing the
  # whole train/test set through DINO in one shot would blow out GPU memory
  results = []
  total = len(imgs)
  for i in range(0, total, chunk_size):
    chunk = imgs[i:i + chunk_size]
    results.append(segment_batch(chunk))
    print(f"  segmented {min(i + chunk_size, total)}/{total}")
  return np.concatenate(results, axis=0)

In [ ]:
def build_model_cifar(latent_dim=LATENT_DIM, num_classes=10):
    # Dual-head CNN: a shared conv encoder feeds both a classification
    # head and a reconstruction/decoder head (predicts the input image
    # back). The reconstruction loss acts as an auxiliary self-supervised
    # signal alongside classification.
    tf.random.set_seed(SEED)
    np.random.seed(SEED)
    img_in = Input(shape=(32, 32, 3), name='image_input')

    # shared encoder (lines 2-3 of Algorithm 1): 32x32 -> 16x16 -> 8x8
    x = layers.Conv2D(32, (3, 3), padding='same', use_bias=False)(img_in)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(32, (3, 3), padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D((2, 2))(x)   # 32x32 -> 16x16

    x = layers.Conv2D(64, (3, 3), padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(64, (3, 3), padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D((2, 2))(x)   # 16x16 -> 8x8

    x = layers.Flatten()(x)
    x = layers.Dense(latent_dim, use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)     # latent vector

    # classification head  p(x) in proposal
    dx = layers.Dense(64, use_bias=False)(x)
    dx = layers.BatchNormalization()(dx)
    dx = layers.Activation('relu')(dx)
    dx = layers.Dropout(0.3)(dx)
    class_out = layers.Dense(num_classes, activation='sigmoid', name='class_output')(dx)

    # decoder / identity head  p'(x) in proposal
    gx = layers.Dense(8 * 8 * 64, use_bias=False)(x)
    gx = layers.BatchNormalization()(gx)
    gx = layers.Activation('relu')(gx)
    gx = layers.Reshape((8, 8, 64))(gx)
    gx = layers.Conv2DTranspose(64, (3, 3), strides=(2, 2), padding='same', use_bias=False)(gx)
    gx = layers.BatchNormalization()(gx)
    gx = layers.Activation('relu')(gx)   # 8x8 -> 16x16
    gx = layers.Conv2DTranspose(32, (3, 3), strides=(2, 2), padding='same', use_bias=False)(gx)
    gx = layers.BatchNormalization()(gx)
    gx = layers.Activation('relu')(gx)   # 16x16 -> 32x32
    recon_out = layers.Conv2D(3, (3, 3), padding='same', activation='sigmoid',
                              name='decoder_output')(gx)

    model = Model(inputs=img_in, outputs=[class_out, recon_out], name='MultitaskBaseline')
    model.compile(
        optimizer=Adam(LR),
        loss={
            'class_output'  : 'sparse_categorical_crossentropy',
            'decoder_output': 'binary_crossentropy',
        },
        loss_weights={
            'class_output'  : 1.0,
            'decoder_output': BETA / 1024.0,  # normalise by pixel count so BETA=1 = equal weight
        },
        metrics={
            'class_output'  : ['accuracy'],
            'decoder_output': ['mse'],
        },
    )
    return model

In [ ]:
def _wrap(gen):
    """Add decoder target (= input image) to each batch."""
    for imgs, labels in gen:
        yield imgs, {'class_output': labels, 'decoder_output': imgs}


def train_model(model, train_gen, val_gen, epochs, steps_per_epoch=STEPS_PER_EPOCH, val_steps=VAL_STEPS, callbacks=None):
    # thin wrapper around model.fit() that wires up the generators
    # (via _wrap, so the decoder target gets attached) and the LR schedule
    return model.fit(
        _wrap(train_gen),
        steps_per_epoch=steps_per_epoch,
        validation_data=_wrap(val_gen),
        validation_steps=val_steps,
        epochs=epochs,
        callbacks=callbacks or [],
        verbose=1,
    )

In [ ]:
def evaluate(model, x_test, y_test):
    # reports classification accuracy + reconstruction MSE on the first
    # 2000 test images
    print('\nevaluating on test set...')
    imgs, labels = x_test[:2000], y_test[:2000]
    class_probs, recons = model.predict(imgs, batch_size=128, verbose=0)
    preds     = np.argmax(class_probs, axis=1)
    accuracy  = float(np.mean(preds == labels))
    recon_mse = float(np.mean((imgs - recons) ** 2))
    print(f'  accuracy  : {accuracy * 100:.2f}%')
    print(f'  recon mse : {recon_mse:.6f}')
    return accuracy, recon_mse

In [ ]:
def plot_reconstructions(model, x_test, y_test, n=8, save_prefix=''):
    # visual sanity check: original image on top, model's reconstruction
    # below it, with the predicted label colored green (correct) or red (wrong)
    idx      = np.random.choice(len(x_test), 8, replace=False)
    sample_x = x_test[idx]
    sample_y = y_test[idx]
    class_probs, recons = model.predict(sample_x, verbose=0)
    preds = np.argmax(class_probs, axis=1)

    fig, axes = plt.subplots(2, n, figsize=(2.5 * n, 5))
    fig.suptitle('original (top) vs reconstruction (bottom) -- green=correct  red=wrong', fontsize=11)
    for col in range(n):
        orig, recon = sample_x[col], recons[col]
        pred, true  = preds[col], sample_y[col]
        mse = float(np.mean((orig - recon) ** 2))
        axes[0, col].imshow(orig)
        axes[0, col].set_title(str(true), fontsize=8)
        axes[0, col].axis('off')
        axes[1, col].imshow(recon)
        color = 'green' if pred == true else 'red'
        axes[1, col].set_title(f'{pred}  mse={mse:.4f}', fontsize=7, color=color)
        axes[1, col].axis('off')
    axes[0, 0].set_ylabel('original',      fontsize=9)
    axes[1, 0].set_ylabel('reconstructed', fontsize=9)
    plt.tight_layout()
    if save_prefix:
        path = os.path.join(OUT_DIR, f'{save_prefix}cifar_reconstructions.png')
        plt.savefig(path, dpi=150, bbox_inches='tight')
        print(f'saved -> {path}')
    plt.show()

In [ ]:
def plot_history(history, save_prefix=''):
    # 3-panel training curve plot: total loss, classification accuracy,
    # reconstruction MSE -- train vs. validation for each
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    fig.suptitle('training history', fontsize=12)

    axes[0].plot(history.history['loss'],     label='train')
    axes[0].plot(history.history['val_loss'], label='val')
    axes[0].set_title('total loss'); axes[0].set_xlabel('epoch'); axes[0].legend()

    axes[1].plot(history.history['class_output_accuracy'],     label='train')
    axes[1].plot(history.history['val_class_output_accuracy'], label='val')
    axes[1].set_title('classification accuracy')
    axes[1].set_xlabel('epoch'); axes[1].set_ylim(0, 1); axes[1].legend()

    axes[2].plot(history.history['decoder_output_mse'],     label='train')
    axes[2].plot(history.history['val_decoder_output_mse'], label='val')
    axes[2].set_title('reconstruction mse'); axes[2].set_xlabel('epoch'); axes[2].legend()

    plt.tight_layout()
    if save_prefix:
        path = os.path.join(OUT_DIR, f'{save_prefix}cifar_training_curves.png')
        plt.savefig(path, dpi=150, bbox_inches='tight')
        print(f'saved -> {path}')
    plt.show()

# Run for CIFAR-10 comparison
#x_train, x_test, y_train, y_test = load_cifar()

In [ ]:
import pickle
import tarfile

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
def load_cifar_from_drive(tar_path):
    # loads CIFAR-10 from a local tar.gz on Drive instead of re-downloading
    # it via Keras' fetcher -- decodes the raw pickled batch files directly
    print("loading cifar from drive...")
    with tarfile.open(tar_path, 'r:gz') as tar:
        tar.extractall('/content/cifar10_manual/')

    def load_batch(path):
        with open(path, 'rb') as f:
            d = pickle.load(f, encoding='bytes')
        return d[b'data'].reshape(-1, 3, 32, 32).transpose(0,2,3,1), np.array(d[b'labels'])

    x_train_list, y_train_list = [], []
    for i in range(1, 6):
        x, y = load_batch(f'/content/cifar10_manual/cifar-10-batches-py/data_batch_{i}')
        x_train_list.append(x)
        y_train_list.append(y)

    x_train = np.concatenate(x_train_list).astype('float32') / 255.0
    y_train = np.concatenate(y_train_list).astype('int32')
    x_test, y_test = load_batch('/content/cifar10_manual/cifar-10-batches-py/test_batch')
    x_test = x_test.astype('float32') / 255.0
    y_test = y_test.astype('int32')

    print(f'train: {x_train.shape}   test: {x_test.shape}')
    return x_train, x_test, y_train, y_test

x_train, x_test, y_train, y_test = load_cifar_from_drive(
    '/content/drive/MyDrive/Adversarial Robustness Research/CIFAR Code/cifar-10-python.tar.gz'
)

In [ ]:
from scipy.ndimage import gaussian_filter

In [ ]:
# quick preview: run segmentation on 8 sample images to visually
# sanity-check the DINO foreground mask before doing the full dataset
test_sample = x_train[:8]
test_seg = segment_batch(test_sample)

In [ ]:
fig, axes = plt.subplots(2, 8, figsize=(20, 5))
fig.suptitle('original (top) vs DINO segmented - blur background (bottom)', fontsize=12)
for col in range(8):
    axes[0, col].imshow(np.clip(x_train[col], 0, 1))
    axes[0, col].axis('off')
    axes[0, col].set_title('original', fontsize=7)
    axes[1, col].imshow(np.clip(test_seg[col], 0, 1))
    axes[1, col].axis('off')
    axes[1, col].set_title('segmented', fontsize=7)
plt.tight_layout()
plt.show()

### Shortcut: load pre-segmented arrays
Run this cell to load already-segmented arrays from Drive instead of re-running DINO segmentation over the full dataset, then skip ahead to **section 3, Post-Segmentation**.

In [ ]:
# Shortcut: load pre-segmented arrays instead of re-running DINO
# segmentation over the whole dataset (slow) -- skip straight to
# Post-Segmentation below if you use this.
from google.colab import drive
drive.mount('/content/drive')
x_train_seg = np.load('/content/drive/MyDrive/Adversarial Robustness Research/CIFAR Code/x_train_seg_cifar.npy')
x_test_seg = np.load('/content/drive/MyDrive/Adversarial Robustness Research/CIFAR Code/x_test_seg_cifar.npy')
print("segmented arrays loaded")

## 2. Full Segmentation (slow path)
Runs DINO segmentation over the entire train/test sets from scratch, then saves the result to Drive. Skip this if you used the shortcut above.

In [ ]:
# slow path: run DINO segmentation over the full train/test sets
print("pre-segmenting training set...")
x_train_seg = segment_all(x_train)
print("pre-segmenting test set...")
x_test_seg = segment_all(x_test)
print("done — segmented images ready")

### Save segmented arrays to Drive

In [ ]:
# save segmented arrays so we don't have to run the segmenter again
from google.colab import drive
drive.mount('/content/drive')
np.save('/content/drive/MyDrive/Adversarial Robustness Research/CIFAR Code/x_train_seg_cifar.npy', x_train_seg)
np.save('/content/drive/MyDrive/Adversarial Robustness Research/CIFAR Code/x_test_seg_cifar.npy', x_test_seg)
print("segmented arrays saved")

## 3. Post-Segmentation: Batch Generators & Candidate Generation
Defines the generators used for training (baseline on-the-fly augmentation, fixed validation, and the mixed approved/anchor generator used by the human + auto models), plus candidate generation and class-balancing for the human review step.

In [ ]:
def make_batches_cifar(x, y, x_seg, batch_size=BATCH_SIZE, m=M_VIEWS):
    """Infinite generator using vectorized segment + augment (lines 6-10 of Algorithm 1)."""
    # each yielded batch = batch_size segmented originals + m augmented
    # copies of each -> batch_size * (m + 1) images per step, shuffled together
    rng = np.random.default_rng(SEED)
    n   = len(x)
    while True:
        idx        = rng.integers(0, n, size=batch_size)
        orig       = x_seg[idx]
        aug_views  = [augment_cifar(orig, rng) for _ in range(m)]
        imgs       = np.concatenate([orig] + aug_views, axis=0)
        labels     = np.tile(y[idx].flatten(), m + 1)
        perm       = rng.permutation(len(imgs))
        yield imgs[perm], labels[perm]

from google.colab import drive
drive.mount('/content/drive')
APPROVED_PATH = '/content/drive/MyDrive/Adversarial Robustness Research/CIFAR Code/approved_seg.npz'

In [ ]:
def generate_candidates():
    # for N_ORIGINALS random training images, generate M_VIEWS augmented
    # copies each -- these are the raw candidates the human (or auto-filter)
    # will approve/reject before they're mixed into human-in-the-loop training
    os.makedirs(CAND_DIR, exist_ok=True)
    rng = np.random.default_rng(SEED)
    source_indices = rng.choice(len(x_train), size=N_ORIGINALS, replace=False)
    src_imgs = x_train_seg[source_indices]
    src_labels = y_train[source_indices].flatten()
    cand_imgs, cand_labels, cand_source_idx, cand_orig_imgs = [], [], [], []
    total = N_ORIGINALS * M_VIEWS
    print(f'generating {N_ORIGINALS} originals x {M_VIEWS} views = {total} candidates...')
    for j, src_idx in enumerate(source_indices):
        orig = src_imgs[j]
        label = src_labels[j]
        for _ in range(M_VIEWS):
            aug = augment_cifar(orig[np.newaxis], rng)[0]
            cand_imgs.append(aug)
            cand_labels.append(label)
            cand_source_idx.append(src_idx)
            cand_orig_imgs.append(orig)
    np.savez(CAND_PATH,
             imgs=np.array(cand_imgs),
             labels=np.array(cand_labels),
             source_idx=np.array(cand_source_idx),
             orig_imgs=np.array(cand_orig_imgs))
    print(f'saved {total} candidates to {CAND_PATH}')

In [ ]:
def make_fixed_val(x_seg, y, batch_size=BATCH_SIZE):
    # deterministic (non-random) validation generator -- cycles through
    # the data in a fixed order so validation numbers are stable/comparable
    # across epochs, unlike the shuffled training generators
    while True:
        for start in range(0, len(x_seg), batch_size):
            imgs = x_seg[start:start+batch_size]
            labels = y[start:start+batch_size]
            yield imgs, labels

In [ ]:
def make_approved_batches(x_train, y_train, x_train_seg, approved_path, batch_size=BATCH_SIZE):
    # generator used to train the human-in-the-loop (and auto-filtered)
    # models: each batch mixes three sources of images (see split below)
    raw = np.load(approved_path)
    aug_imgs = raw['imgs']
    aug_labels = raw['labels']
    n_aug = len(aug_imgs)
    n_train = len(x_train)
    rng = np.random.default_rng(SEED)

    # batch is split three ways:
    # 1/2 regular augmented originals (same as baseline)
    # 1/4 segmented originals as anchors
    # 1/4 human-approved samples as bonus signal
    third = max(1, batch_size // 4)
    half = batch_size - (third * 2)

    while True:
        # regular augmented batch — same as baseline
        idx = rng.integers(0, n_train, size=half)
        orig = x_train_seg[idx]
        aug_views = augment_cifar(orig, rng)
        orig_labs = y_train[idx].flatten()

        # anchor originals
        anchor_idx = rng.integers(0, n_train, size=third)
        anchors = x_train_seg[anchor_idx]
        anchor_labs = y_train[anchor_idx].flatten()

        # human approved bonus
        aug_idx = rng.integers(0, n_aug, size=third)
        approved = augment_cifar(aug_imgs[aug_idx], rng)
        approved_labs = aug_labels[aug_idx]

        imgs = np.concatenate([aug_views, anchors, approved], axis=0)
        labels = np.concatenate([orig_labs, anchor_labs, approved_labs], axis=0)
        perm = rng.permutation(len(imgs))
        yield imgs[perm], labels[perm]

In [ ]:
def balance_approved(approved_path):
    # after the human review widget saves approved_path, this caps every
    # class at the size of the smallest class -- so the human-approved set
    # isn't skewed toward whichever classes got approved more often
    raw = np.load(approved_path)
    imgs = raw['imgs']
    labels = raw['labels']

    min_count = min(np.sum(labels == i) for i in range(10))
    print(f"Balancing to {min_count} samples per class")

    balanced_idx = []
    for i in range(10):
        class_idx = np.where(labels == i)[0]
        selected = np.random.choice(class_idx, size=min_count, replace=False)
        balanced_idx.extend(selected)

    balanced_idx = np.array(balanced_idx)
    np.savez_compressed(approved_path,
                        imgs=imgs[balanced_idx],
                        labels=labels[balanced_idx])
    print(f"Balanced approved set: {len(balanced_idx)} total samples")

balance_approved(APPROVED_PATH)

## 4. Part 1 — Baseline Model
Trains the baseline model (on-the-fly augmentation only, no human/auto filtering) and reports its per-class accuracy.

In [ ]:
def generate_auto_approved(candidates_path, baseline_model,
                           threshold=0.7, target_per_class=469):
    # cheap stand-in for the human reviewer: uses the baseline model's
    # own predictions/confidence to auto-approve candidates instead
    raw = np.load(candidates_path)
    cand_imgs = raw['imgs']
    cand_labels = raw['labels']

    # run candidates through baseline model
    print("scoring candidates with baseline model...")
    class_probs, _ = baseline_model.predict(cand_imgs,
                                             batch_size=128,
                                             verbose=0)

    # keep candidates where model predicts correct label
    # with confidence >= threshold
    pred_labels = np.argmax(class_probs, axis=1)
    pred_conf = np.max(class_probs, axis=1)

    correct_pred = pred_labels == cand_labels
    high_conf = pred_conf >= threshold
    auto_mask = correct_pred & high_conf

    auto_imgs = cand_imgs[auto_mask]
    auto_labels = cand_labels[auto_mask]

    print(f"auto-approved: {auto_mask.sum()} / {len(cand_imgs)} "
          f"({100*auto_mask.mean():.1f}%)")
    print(f"label distribution: {np.bincount(auto_labels)}")

    # balance to target_per_class -- caps each class at target_per_class
    # samples so the auto-approved set isn't skewed toward classes the
    # baseline model is already good at
    rng = np.random.default_rng(SEED)
    balanced_idx = []
    for i in range(10):
        class_idx = np.where(auto_labels == i)[0]
        if len(class_idx) >= target_per_class:
            selected = rng.choice(class_idx,
                                  size=target_per_class,
                                  replace=False)
        else:
            # if not enough, take all and note the shortfall
            selected = class_idx
            print(f"  warning: class {i} only has "
                  f"{len(class_idx)} samples")
        balanced_idx.extend(selected)

    balanced_idx = np.array(balanced_idx)

    AUTO_PATH = os.path.join(CAND_DIR, 'auto_approved.npz')
    np.savez_compressed(AUTO_PATH,
                        imgs=auto_imgs[balanced_idx],
                        labels=auto_labels[balanced_idx])
    print(f"saved {len(balanced_idx)} auto-approved samples")
    return AUTO_PATH

raw = np.load(APPROVED_PATH)
n_approved = len(raw['labels'])
print(f"Approved set size: {n_approved}")

In [ ]:
print('=' * 60)
print('PART 1 — BASELINE MODEL  (on-the-fly augmentation)')
print('=' * 60)

In [ ]:
# match the baseline's training-subset size to n_approved, so baseline
# and human-in-the-loop models see the same *number* of custom images
# -- keeps the comparison fair
rng_fixed = np.random.default_rng(SEED)
subset_idx = rng_fixed.choice(len(x_train), size=n_approved, replace=False)
x_train_subset = x_train_seg[subset_idx]
y_train_subset = y_train[subset_idx]

In [ ]:
baseline_model = build_model_cifar()
baseline_model.summary()

In [ ]:
baseline_history = train_model(
    baseline_model,
    make_batches_cifar(x_train_subset, y_train_subset, x_train_subset),
    make_fixed_val(x_test_seg, y_test),
    epochs=EPOCHS_BASELINE,
    callbacks=[make_lr_schedule(EPOCHS_BASELINE, warmup=5)],
)

In [ ]:
baseline_acc, baseline_mse = evaluate(baseline_model, x_test_seg, y_test)
results['baseline'] = {'accuracy': baseline_acc, 'recon_mse': baseline_mse}

In [ ]:
plot_reconstructions(baseline_model, x_test, y_test, n=8, save_prefix='baseline_')
plot_history(baseline_history, save_prefix='baseline_')

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
_imgs, _labels = x_test[:2000], y_test[:2000]
_probs, _ = baseline_model.predict(_imgs, batch_size=128, verbose=0)
_preds = np.argmax(_probs, axis=1)

In [ ]:
cifar_classes = ['airplane', 'automobile', 'bird', 'cat', 'deer',
                 'dog', 'frog', 'horse', 'ship', 'truck']

print('Baseline — per-class accuracy')
print('-' * 36)
for class_idx in range(10):
    mask = _labels == class_idx
    acc  = np.mean(_preds[mask] == _labels[mask]) * 100
    print(f'  {cifar_classes[class_idx]:<12} : {acc:6.2f}%  ({mask.sum()} samples)')

# full classification report
print('\nBaseline — classification report')
print(classification_report(_labels, _preds, target_names=[str(i) for i in range(10)]))

## 5. Part 2 — Human-in-the-Loop Model
### Shortcut: load pre-approved candidates
Run the cell below to load a previously-approved candidate set from Drive instead of redoing the manual review, then skip to **Step 2**.

In [ ]:
# Shortcut: load previously-approved candidates from Drive instead
# of redoing the manual review widget below
from google.colab import drive
drive.mount('/content/drive')
APPROVED_PATH = '/content/drive/MyDrive/Adversarial Robustness Research/CIFAR Code/approved_seg.npz'
print(f"approved candidates loaded from {APPROVED_PATH}")
raw = np.load(APPROVED_PATH)
n_approved = len(raw['labels'])
print(f"Approved set size: {n_approved}")

### Step 1 — Generate & Review Candidates
Generates fresh augmented candidates and launches the interactive review widget (approve/reject each one). Skip this if you loaded pre-approved candidates above.

In [ ]:
import os
# clear old review state so widget starts fresh
if os.path.exists(STATE_PATH):
    os.remove(STATE_PATH)
    print("cleared old review state")
if os.path.exists(CAND_PATH):
    os.remove(CAND_PATH)
    print("cleared old candidates")
generate_candidates()

In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output
except ImportError:
    raise ImportError("ipywidgets is required for the review UI.  Run:  pip install ipywidgets")

if not os.path.exists(CAND_PATH):
    print('ERROR: candidates.npz not found. Run the Generate Candidates cell first.')
else:
    _raw  = np.load(CAND_PATH)
    _data = {k: _raw[k] for k in _raw.files}
    _n    = len(_data['imgs'])

    # load or initialise review state -- lets you close the tab and come
    # back later without losing approve/reject progress
    if os.path.exists(STATE_PATH):
        with open(STATE_PATH) as f:
            _saved = json.load(f)
        _rev = {'idx': _saved['current_idx'], 'decisions': list(_saved['decisions'])}
        print(f'Resuming — {_rev["idx"]} of {_n} decisions already recorded.')
    else:
        _rev = {'idx': 0, 'decisions': []}
        print(f'Starting fresh review of {_n} candidates.')

    # ── widget helpers ─────────────────────────────────────────────

    def _rev_save():
        # persist review progress to disk after every single decision
        state = {'current_idx': _rev['idx'], 'decisions': _rev['decisions']}
        with open(STATE_PATH, 'w') as f:
            json.dump(state, f, indent=2)

    def _rev_finalize():
        # called once every candidate has been reviewed: writes out only
        # the approved images/labels to APPROVED_PATH
        decisions  = _rev['decisions']
        n_dec      = len(decisions)
        n_app      = sum(1 for d in decisions if d == 1)
        approved_i = [i for i, d in enumerate(decisions) if d == 1]
        if approved_i:
            np.savez_compressed(APPROVED_PATH,
                                imgs=_data['imgs'][approved_i],
                                labels=_data['labels'][approved_i])
        pct = 100 * n_app / n_dec if n_dec else 0
        lbl_status.value = (
            f'<b>Done!</b>  Reviewed: {n_dec}  '
            f'| <span style="color:green">Approved: {n_app} ({pct:.0f}%)</span>  '
            f'| <span style="color:red">Rejected: {n_dec - n_app}</span>  '
            f'<br><b>approved_seg.npz saved.</b>  Ready to train — run Step 3.')

    # ── widgets ────────────────────────────────────────────────────

    _W = widgets.Layout
    out_img    = widgets.Output()
    lbl_prog   = widgets.HTML()
    lbl_status = widgets.HTML()
    btn_a      = widgets.Button(description='✓ Approve',    button_style='success', layout=_W(width='130px'))
    btn_r      = widgets.Button(description='✗ Reject',     button_style='danger',  layout=_W(width='130px'))
    btn_u      = widgets.Button(description='↩ Undo',       button_style='warning', layout=_W(width='130px'))
    btn_done   = widgets.Button(description='💾 Save & Done', button_style='info',  layout=_W(width='140px'))

    def _rev_update_ui():
        # redraws the progress label and the current original-vs-augmented
        # image pair inside out_img
        idx       = _rev['idx']
        decisions = _rev['decisions']
        n_dec     = len(decisions)
        n_app     = sum(1 for d in decisions if d == 1)

        lbl_prog.value = (
            f'<b>Candidate {idx + 1} / {_n}</b> &nbsp;&nbsp; '
            f'Approved: <b style="color:green">{n_app}</b> &nbsp; '
            f'Rejected: <b style="color:red">{n_dec - n_app}</b> &nbsp; '
            f'Remaining: <b>{_n - idx}</b>')

        with out_img:
            clear_output(wait=True)
            if idx >= _n:
                print('All candidates reviewed!')
                return
            img_orig = _data['orig_imgs'][idx]
            img_aug  = _data['imgs'][idx]
            label    = int(_data['labels'][idx])
            fig, axes = plt.subplots(1, 2, figsize=(5, 2.8))
            axes[0].imshow(np.clip(img_orig, 0, 1))
            cifar_classes = ['airplane', 'automobile', 'bird', 'cat', 'deer',
                 'dog', 'frog', 'horse', 'ship', 'truck']
            axes[0].set_title(f'Original  ({cifar_classes[label]})', fontsize=9)
            axes[0].axis('off')
            axes[1].imshow(np.clip(img_aug, 0, 1))
            axes[1].set_title('Augmented candidate', fontsize=9)
            axes[1].axis('off')
            plt.tight_layout()
            plt.show()

    # ── button callbacks ──────────────────────────────────────────
    def _on_approve(b):
        if _rev['idx'] >= _n: return
        _rev['decisions'].append(1)
        _rev['idx'] += 1
        _rev_save()
        lbl_status.value = '<span style="color:green">✓ Approved</span>'
        if _rev['idx'] >= _n:
            _rev_finalize()
        else:
            _rev_update_ui()

    def _on_reject(b):
        if _rev['idx'] >= _n: return
        _rev['decisions'].append(-1)
        _rev['idx'] += 1
        _rev_save()
        lbl_status.value = '<span style="color:red">✗ Rejected</span>'
        if _rev['idx'] >= _n:
            _rev_finalize()
        else:
            _rev_update_ui()

    def _on_undo(b):
        # pop the last decision and step back one candidate
        if not _rev['decisions']: return
        _rev['decisions'].pop()
        _rev['idx'] = max(0, _rev['idx'] - 1)
        _rev_save()
        lbl_status.value = '↩ Undone'
        _rev_update_ui()

    def _on_done(b):
        # manual early-exit: save + finalize whatever's been reviewed so far
        _rev_save()
        _rev_finalize()

    btn_a.on_click(_on_approve)
    btn_r.on_click(_on_reject)
    btn_u.on_click(_on_undo)
    btn_done.on_click(_on_done)

    ui = widgets.VBox([
        lbl_prog,
        out_img,
        widgets.HBox([btn_a, btn_r, btn_u, btn_done]),
        lbl_status,
    ])
    display(ui)
    _rev_update_ui()

### Save approved candidates to Drive

In [ ]:
# copy approved candidates to Drive so they persist across sessions
import shutil
shutil.copy(APPROVED_PATH, '/content/drive/MyDrive/Adversarial Robustness Research/CIFAR Code/approved_seg.npz')
print("approved candidates saved to Drive")

### Step 2 — Train the Human-in-the-Loop Model

In [ ]:
raw = np.load(APPROVED_PATH)
print(f"approved imgs shape: {raw['imgs'].shape}")
print(f"approved imgs min: {raw['imgs'].min():.3f}  max: {raw['imgs'].max():.3f}")
print(f"x_train_seg min: {x_train_seg.min():.3f}  max: {x_train_seg.max():.3f}")

In [ ]:
if not os.path.exists(APPROVED_PATH):
    print('Complete the review widget first.')
else:
    print('=' * 60)
    print('PART 2 — HUMAN-IN-THE-LOOP MODEL')
    print('=' * 60)
    human_model = build_model_cifar()
    human_model.summary()
human_history = train_model(
    human_model,
    make_approved_batches(x_train, y_train, x_train_seg, APPROVED_PATH),
    make_fixed_val(x_test_seg, y_test),
    epochs=EPOCHS_HUMAN,
    callbacks=[make_lr_schedule(EPOCHS_HUMAN, warmup=2)],
)

In [ ]:
human_acc, human_mse = evaluate(human_model, x_test_seg, y_test)
results['human'] = {'accuracy': human_acc, 'recon_mse': human_mse}
plot_reconstructions(human_model, x_test, y_test, n=8, save_prefix='human_')
plot_history(human_history, save_prefix='human_')

## 6. Part 3 — Auto-Filtered Model
Instead of a human reviewer, the baseline model itself auto-approves high-confidence candidates. Trains that model and reports a quick three-way (baseline/auto/human) comparison.

In [ ]:
generate_candidates()

In [ ]:
AUTO_PATH = generate_auto_approved(CAND_PATH, baseline_model,
                                    threshold=0.7)

auto_model = build_model_cifar()
auto_history = train_model(
    auto_model,
    make_approved_batches(x_train, y_train,
                          x_train_seg, AUTO_PATH),
    make_fixed_val(x_test_seg, y_test),
    epochs=EPOCHS_BASELINE,
    callbacks=[make_lr_schedule(EPOCHS_BASELINE, warmup=5)],
)

In [ ]:
auto_acc, auto_mse = evaluate(auto_model, x_test_seg, y_test)

## 7. Comparison — Baseline vs. Human vs. Auto
Detailed baseline-vs-human summary table and bar chart, per-class diagnostic breakdown, and the three-way comparison used later.

In [ ]:
if len(results) < 2:
    missing = [k for k in ('baseline', 'human') if k not in results]
    print(f'Missing results for: {missing}. Run the corresponding Part first.')
else:
    b = results['baseline']
    h = results['human']

    # ── text summary ───────────────────────────────────────────────
    print('=' * 55)
    print(f'{"":30s}  {"Baseline":>10s}  {"Human":>10s}')
    print('-' * 55)
    print(f'{"Test accuracy":30s}  {b["accuracy"]*100:>9.2f}%  {h["accuracy"]*100:>9.2f}%')
    print(f'{"Reconstruction MSE":30s}  {b["recon_mse"]:>10.6f}  {h["recon_mse"]:>10.6f}')
    delta_acc = (h['accuracy'] - b['accuracy']) * 100
    delta_mse = h['recon_mse'] - b['recon_mse']
    direction_acc = 'improvement' if delta_acc > 0 else 'decline'
    direction_mse = 'improvement' if delta_mse < 0 else 'increase'
    print('-' * 55)
    print(f'  Accuracy delta : {delta_acc:+.2f}%  ({direction_acc})')
    print(f'  MSE delta      : {delta_mse:+.6f}  ({direction_mse})')
    print('=' * 55)

    # ── bar chart ──────────────────────────────────────────────────
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 4))
    fig.suptitle('Baseline vs. Human-in-the-Loop', fontsize=13, fontweight='bold')

    labels = ['Baseline', 'Human']
    colors = ['#4c72b0', '#dd8452']

    accs = [b['accuracy'] * 100, h['accuracy'] * 100]
    bars1 = ax1.bar(labels, accs, color=colors, width=0.5)
    ax1.set_ylabel('Test Accuracy (%)')
    ax1.set_title('Classification Accuracy')
    ax1.set_ylim(max(0, min(accs) - 5), min(100, max(accs) + 5))
    for bar, v in zip(bars1, accs):
        ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.2,
                 f'{v:.2f}%', ha='center', va='bottom', fontsize=11)

    mses = [b['recon_mse'], h['recon_mse']]
    bars2 = ax2.bar(labels, mses, color=colors, width=0.5)
    ax2.set_ylabel('Reconstruction MSE')
    ax2.set_title('Reconstruction MSE  (lower = better)')
    ax2.set_ylim(0, max(mses) * 1.25)
    for bar, v in zip(bars2, mses):
        ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + max(mses) * 0.01,
                 f'{v:.5f}', ha='center', va='bottom', fontsize=10)

    plt.tight_layout()
    cmp_path = os.path.join(OUT_DIR, 'comparison.png')
    plt.savefig(cmp_path, dpi=150, bbox_inches='tight')
    print(f'\ncomparison chart saved -> {cmp_path}')
    plt.show()

# ── quick three-way accuracy comparison: baseline vs auto vs human ──
print('=' * 55)
print(f'{"":30s}  {"Baseline":>10s}  {"Auto":>10s}  {"Human":>10s}')
print('-' * 55)
print(f'{"Test accuracy":30s}  {baseline_acc*100:>9.2f}%  {auto_acc*100:>9.2f}%  {human_acc*100:>9.2f}%')
print(f'{"Reconstruction MSE":30s}  {baseline_mse:>10.6f}  {auto_mse:>10.6f}  {human_mse:>10.6f}')
print('-' * 55)
print(f'{"Delta vs Baseline (Acc)":30s}  {"—":>10s}  {(auto_acc-baseline_acc)*100:>+9.2f}%  {(human_acc-baseline_acc)*100:>+9.2f}%')
print(f'{"Delta vs Auto (Acc)":30s}  {"—":>10s}  {"—":>10s}  {(human_acc-auto_acc)*100:>+9.2f}%')
print('=' * 55)

In [ ]:
# ── Diagnostic: per-class accuracy comparison ──────────────────────
cifar_classes = ['airplane', 'automobile', 'bird', 'cat', 'deer',
                 'dog', 'frog', 'horse', 'ship', 'truck']

x_diag = x_test_seg[:2000]
y_diag = y_test[:2000]

In [ ]:
base_probs, base_recons = baseline_model.predict(x_diag, batch_size=128, verbose=0)
human_probs, human_recons = human_model.predict(x_diag, batch_size=128, verbose=0)

In [ ]:
base_preds = np.argmax(base_probs, axis=1)
human_preds = np.argmax(human_probs, axis=1)

In [ ]:
print(f"{'Class':<12} {'Base Acc':>10} {'Human Acc':>10} {'Delta':>10} {'Base MSE':>10} {'Human MSE':>10}")
print("-" * 65)
for i, cls in enumerate(cifar_classes):
    mask = y_diag == i
    base_acc = np.mean(base_preds[mask] == y_diag[mask]) * 100
    human_acc = np.mean(human_preds[mask] == y_diag[mask]) * 100
    base_mse = np.mean((x_diag[mask] - base_recons[mask]) ** 2)
    human_mse = np.mean((x_diag[mask] - human_recons[mask]) ** 2)
    delta = human_acc - base_acc
    print(f"{cls:<12} {base_acc:>9.1f}% {human_acc:>9.1f}% {delta:>+9.1f}% {base_mse:>10.4f} {human_mse:>10.4f}")

# ── confusion: what is human model confusing? ──────────────────────
print("\nClasses where human model is WORSE by >5%:")
for i, cls in enumerate(cifar_classes):
    mask = y_diag == i
    base_acc = np.mean(base_preds[mask] == y_diag[mask]) * 100
    human_acc = np.mean(human_preds[mask] == y_diag[mask]) * 100
    if human_acc - base_acc < -5:
        # what is human model predicting instead?
        human_wrong = human_preds[mask][human_preds[mask] != y_diag[mask]]
        if len(human_wrong) > 0:
            most_confused = cifar_classes[np.bincount(human_wrong).argmax()]
            print(f"  {cls} → most confused with: {most_confused}")

raw = np.load(APPROVED_PATH)
approved_labels = raw['labels']
print("Approved set class distribution:")
for i, cls in enumerate(cifar_classes):
    count = np.sum(approved_labels == i)
    pct = 100 * count / len(approved_labels)
    print(f"  {cls:<12}: {count:4d} ({pct:.1f}%)")

## 8. Part 4 — Adversarial Evaluation
FGSM and PGD attacks swept across a range of epsilon values against all three models (baseline, auto-filtered, human-in-the-loop).

In [ ]:
# ── Part 4 — Adversarial Evaluation (FGSM) ──────────────────────
# Run after completing both Part 1 and Part 2.

In [ ]:
def fgsm_attack(model, images, labels, epsilon):
    # FGSM (Fast Gradient Sign Method): single-step attack that nudges
    # each pixel by `epsilon` in the direction that most increases the
    # loss, then clips back into valid [0, 1] pixel range
    import tensorflow as tf
    images = tf.cast(images, tf.float32)
    with tf.GradientTape() as tape:
        tape.watch(images)
        preds, _ = model(images, training=False)
        loss = keras.losses.sparse_categorical_crossentropy(labels, preds)
    gradients = tape.gradient(loss, images)
    perturbations = epsilon * tf.sign(gradients)
    return tf.clip_by_value(images + perturbations, 0, 1)

epsilons = [0.0, 0.005, 0.01, 0.015, 0.02, 0.03, 0.04, 0.05, 0.075, 0.1, 0.2, 0.3]
x_eval = tf.cast(x_test_seg[:2000], tf.float32)  # evaluate on a 2000-image subset for speed
y_eval = y_test[:2000]

In [ ]:
baseline_fgsm_accs = []
human_fgsm_accs = []

In [ ]:
for eps in epsilons:
    x_adv = fgsm_attack(baseline_model, x_eval, y_eval, eps)
    preds, _ = baseline_model(x_adv, training=False)
    baseline_fgsm_accs.append(np.mean(np.argmax(preds.numpy(), axis=1) == y_eval))

    x_adv = fgsm_attack(human_model, x_eval, y_eval, eps)
    preds, _ = human_model(x_adv, training=False)
    human_fgsm_accs.append(np.mean(np.argmax(preds.numpy(), axis=1) == y_eval))

def pgd_attack(model, images, labels, epsilon, alpha=0.01, num_steps=40):
    # PGD (Projected Gradient Descent): iterative, stronger version of
    # FGSM -- takes `num_steps` small FGSM-style steps of size `alpha`,
    # projecting back into the epsilon-ball around the original image
    # after each step
    x_adv = tf.identity(images)
    for _ in range(num_steps):
        with tf.GradientTape() as tape:
            tape.watch(x_adv)
            preds, _ = model(x_adv, training=False)
            loss = keras.losses.sparse_categorical_crossentropy(labels, preds)
        gradients = tape.gradient(loss, x_adv)
        x_adv = x_adv + alpha * tf.sign(gradients)
        x_adv = tf.clip_by_value(x_adv, images - epsilon, images + epsilon)
        x_adv = tf.clip_by_value(x_adv, 0, 1)
    return x_adv

baseline_pgd_accs = []
human_pgd_accs = []

In [ ]:
for eps in epsilons:
    x_adv = pgd_attack(baseline_model, x_eval, y_eval, eps)
    preds, _ = baseline_model(x_adv, training=False)
    baseline_pgd_accs.append(np.mean(np.argmax(preds.numpy(), axis=1) == y_eval))

    x_adv = pgd_attack(human_model, x_eval, y_eval, eps)
    preds, _ = human_model(x_adv, training=False)
    human_pgd_accs.append(np.mean(np.argmax(preds.numpy(), axis=1) == y_eval))

In [ ]:
# normalize all to decimals regardless of how they were stored
baseline_acc_n = baseline_acc / 100 if baseline_acc > 1 else baseline_acc
auto_acc_n = auto_acc / 100 if auto_acc > 1 else auto_acc
human_acc_n = human_acc / 100 if human_acc > 1 else human_acc

In [ ]:
# ── Three-way comparison: Baseline vs Auto vs Human ──────────────

In [ ]:
print('=' * 60)
print(f'{"":30s}  {"Baseline":>10s}  {"Auto":>10s}  {"Human":>10s}')
print('-' * 60)
print(f'{"Test accuracy":30s}  {baseline_acc_n*100:>9.2f}%  {auto_acc_n*100:>9.2f}%  {human_acc_n*100:>9.2f}%')
print(f'{"Reconstruction MSE":30s}  {baseline_mse:>10.6f}  {auto_mse:>10.6f}  {human_mse:>10.6f}')
print('-' * 60)
print(f'{"Delta vs Baseline (Acc)":30s}  {"—":>10s}  {(auto_acc_n-baseline_acc_n)*100:>+9.2f}%  {(human_acc_n-baseline_acc_n)*100:>+9.2f}%')
print(f'{"Delta vs Baseline (MSE)":30s}  {"—":>10s}  {(auto_mse-baseline_mse):>+10.6f}  {(human_mse-baseline_mse):>+10.6f}')
print('=' * 60)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
fig.suptitle('Baseline vs. Auto-Filtered vs. Human-in-the-Loop',
             fontsize=13, fontweight='bold')

labels = ['Baseline', 'Auto', 'Human']
colors = ['#4c72b0', '#55a868', '#dd8452']

In [ ]:
accs = [baseline_acc_n*100, auto_acc_n*100, human_acc_n*100]
bars1 = ax1.bar(labels, accs, color=colors, width=0.5)
ax1.set_ylabel('Test Accuracy (%)')
ax1.set_title('Classification Accuracy')
ax1.set_ylim(max(0, min(accs) - 5), min(100, max(accs) + 5))
for bar, v in zip(bars1, accs):
    ax1.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 0.2,
             f'{v:.2f}%', ha='center', va='bottom', fontsize=11)

mses = [baseline_mse, auto_mse, human_mse]
bars2 = ax2.bar(labels, mses, color=colors, width=0.5)
ax2.set_ylabel('Reconstruction MSE')
ax2.set_title('Reconstruction MSE  (lower = better)')
ax2.set_ylim(0, max(mses) * 1.25)
for bar, v in zip(bars2, mses):
    ax2.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + max(mses)*0.01,
             f'{v:.5f}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/Adversarial Robustness Research/CIFAR Code/three_way_comparison.png',
            dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# FGSM three-way
auto_fgsm_accs = []
for eps in epsilons:
    x_adv = fgsm_attack(auto_model, x_eval, y_eval, eps)
    preds, _ = auto_model(x_adv, training=False)
    auto_fgsm_accs.append(np.mean(np.argmax(preds.numpy(), axis=1) == y_eval))

print(f"\n{'Epsilon':<10} {'Baseline':>10} {'Auto':>10} {'Human':>10}")
print("-" * 45)
for eps, b, a, h in zip(epsilons, baseline_fgsm_accs, auto_fgsm_accs, human_fgsm_accs):
    print(f"{eps:<10.3f} {b*100:>9.2f}% {a*100:>9.2f}% {h*100:>9.2f}%")

# PGD three-way
auto_pgd_accs = []
for eps in epsilons:
    x_adv = pgd_attack(auto_model, x_eval, y_eval, eps)
    preds, _ = auto_model(x_adv, training=False)
    auto_pgd_accs.append(np.mean(np.argmax(preds.numpy(), axis=1) == y_eval))

print(f"\n{'Epsilon':<10} {'Baseline':>10} {'Auto':>10} {'Human':>10}")
print("-" * 45)
for eps, b, a, h in zip(epsilons, baseline_pgd_accs, auto_pgd_accs, human_pgd_accs):
    print(f"{eps:<10.3f} {b*100:>9.2f}% {a*100:>9.2f}% {h*100:>9.2f}%")

# Robustness plot
plt.figure(figsize=(10, 5))
plt.plot(epsilons, [a*100 for a in baseline_fgsm_accs], 'o-',
         color='#4c72b0', label='Baseline (FGSM)')
plt.plot(epsilons, [a*100 for a in auto_fgsm_accs], 'o-',
         color='#55a868', label='Auto (FGSM)')
plt.plot(epsilons, [a*100 for a in human_fgsm_accs], 'o-',
         color='#dd8452', label='Human (FGSM)')
plt.plot(epsilons, [a*100 for a in baseline_pgd_accs], 'o--',
         color='#4c72b0', label='Baseline (PGD)')
plt.plot(epsilons, [a*100 for a in auto_pgd_accs], 'o--',
         color='#55a868', label='Auto (PGD)')
plt.plot(epsilons, [a*100 for a in human_pgd_accs], 'o--',
         color='#dd8452', label='Human (PGD)')
plt.xlabel('Epsilon (attack strength)')
plt.ylabel('Accuracy (%)')
plt.title('Adversarial Robustness — Baseline vs. Auto vs. Human (CIFAR-10)')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/Adversarial Robustness Research/CIFAR Code/three_way_robustness.png',
            dpi=150, bbox_inches='tight')
plt.show()

## Checkpoint: Save & Load Custom Model Results

New section (not in the original export). Saves the three trained custom models (baseline, auto-filtered, human-in-the-loop) plus every metric computed in Parts 1-4 to a `checkpoints` folder inside `CIFAR Code` on Drive, so a fresh Colab session can restore everything without retraining. Mirrors the checkpoint pattern already used for the STL-10 notebook.

### Save Checkpoint

In [ ]:
import pickle

SAVE_DIR_CIFAR = '/content/drive/MyDrive/Adversarial Robustness Research/CIFAR Code/checkpoints'
os.makedirs(SAVE_DIR_CIFAR, exist_ok=True)

# save models
baseline_model.save(os.path.join(SAVE_DIR_CIFAR, 'baseline_model.keras'))
auto_model.save(os.path.join(SAVE_DIR_CIFAR, 'auto_model.keras'))
human_model.save(os.path.join(SAVE_DIR_CIFAR, 'human_model.keras'))
print("models saved")

# save results variables
checkpoint = {
    'baseline_acc':          baseline_acc,
    'baseline_mse':          baseline_mse,
    'auto_acc':              auto_acc,
    'auto_mse':              auto_mse,
    'human_acc':             human_acc,
    'human_mse':             human_mse,
    'baseline_fgsm_accs':    baseline_fgsm_accs,
    'baseline_pgd_accs':     baseline_pgd_accs,
    'auto_fgsm_accs':        auto_fgsm_accs,
    'auto_pgd_accs':         auto_pgd_accs,
    'human_fgsm_accs':       human_fgsm_accs,
    'human_pgd_accs':        human_pgd_accs,
    'epsilons':              epsilons,
    'subset_idx':            subset_idx,
}
with open(os.path.join(SAVE_DIR_CIFAR, 'results.pkl'), 'wb') as f:
    pickle.dump(checkpoint, f)
print("results saved")

### Load Checkpoint

Shortcut: restores the saved models + metrics from the `checkpoints` folder instead of retraining from scratch. Run this in place of Parts 1-4 in a fresh session.

In [ ]:
import pickle
from keras.models import load_model

SAVE_DIR_CIFAR = '/content/drive/MyDrive/Adversarial Robustness Research/CIFAR Code/checkpoints'

baseline_model = load_model(os.path.join(SAVE_DIR_CIFAR, 'baseline_model.keras'))
auto_model     = load_model(os.path.join(SAVE_DIR_CIFAR, 'auto_model.keras'))
human_model    = load_model(os.path.join(SAVE_DIR_CIFAR, 'human_model.keras'))
print("models loaded")

with open(os.path.join(SAVE_DIR_CIFAR, 'results.pkl'), 'rb') as f:
    checkpoint = pickle.load(f)

baseline_acc       = checkpoint['baseline_acc']
baseline_mse       = checkpoint['baseline_mse']
auto_acc           = checkpoint['auto_acc']
auto_mse           = checkpoint['auto_mse']
human_acc          = checkpoint['human_acc']
human_mse          = checkpoint['human_mse']
baseline_fgsm_accs = checkpoint['baseline_fgsm_accs']
baseline_pgd_accs  = checkpoint['baseline_pgd_accs']
auto_fgsm_accs     = checkpoint['auto_fgsm_accs']
auto_pgd_accs      = checkpoint['auto_pgd_accs']
human_fgsm_accs    = checkpoint['human_fgsm_accs']
human_pgd_accs     = checkpoint['human_pgd_accs']
epsilons           = checkpoint['epsilons']
subset_idx         = checkpoint['subset_idx']
print("results restored")

# still need to reload manually:
# x_train, x_test, y_train, y_test  — load_cifar_from_drive(...)
# x_train_seg, x_test_seg           — np.load from Drive
# APPROVED_PATH                     — should still exist in candidates/
print("remember to reload: data, segmented arrays, APPROVED_PATH")

## 9. Transfer Learning
Repeats the baseline-vs-human comparison using pretrained ImageNet backbones instead of the custom CNN.
### MobileNetV2

In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mobilenet_preprocess

In [ ]:
print('=' * 60)
print('PART 5 — TRANSFER LEARNING (MobileNetV2)')
print('=' * 60)

In [ ]:
# preprocess once here, never again inside the model
x_train_tl = mobilenet_preprocess(x_train * 255.0)  # -> [-1, 1]
x_test_tl  = mobilenet_preprocess(x_test  * 255.0)

In [ ]:
# also preprocess the segmented arrays for the human generator
x_train_seg_tl = mobilenet_preprocess(x_train_seg * 255.0)
x_test_seg_tl  = mobilenet_preprocess(x_test_seg  * 255.0)

In [ ]:
# preprocess approved candidates once
raw_approved     = np.load(APPROVED_PATH)
approved_imgs_tl = mobilenet_preprocess(raw_approved['imgs'] * 255.0)
approved_labels  = raw_approved['labels']
np.savez('/content/approved_tl.npz',
         imgs=approved_imgs_tl, labels=approved_labels)
APPROVED_PATH_TL = '/content/approved_tl.npz'

In [ ]:
def build_transfer_model(name='MobileNetV2Transfer'):
    mob_base = MobileNetV2(
        weights='imagenet',
        include_top=False,
        input_shape=(32, 32, 3),
        alpha=1.0,
    )


    # freeze everything except the last 35 layers
    for layer in mob_base.layers[:-35]:
        layer.trainable = False

    img_in = Input(shape=(32, 32, 3), name='image_input')
    # NOTE: no preprocessing inside the model — data is already preprocessed
    x = mob_base(img_in, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(0.3)(x)

    # classification head
    class_out = layers.Dense(10, activation='softmax', name='class_output')(x)

    # decoder head — same structure as your custom model
    gx = layers.Dense(8 * 8 * 64, use_bias=False)(x)
    gx = layers.BatchNormalization()(gx)
    gx = layers.Activation('relu')(gx)
    gx = layers.Reshape((8, 8, 64))(gx)
    gx = layers.Conv2DTranspose(64, (3,3), strides=(2,2), padding='same', use_bias=False)(gx)
    gx = layers.BatchNormalization()(gx)
    gx = layers.Activation('relu')(gx)
    gx = layers.Conv2DTranspose(32, (3,3), strides=(2,2), padding='same', use_bias=False)(gx)
    gx = layers.BatchNormalization()(gx)
    gx = layers.Activation('relu')(gx)
    recon_out = layers.Conv2D(3, (3,3), padding='same', activation='sigmoid',
                              name='decoder_output')(gx)

    model = Model(inputs=img_in, outputs=[class_out, recon_out], name=name)
    model.compile(
        optimizer=Adam(LR),
        loss={
            'class_output'  : 'sparse_categorical_crossentropy',
            'decoder_output': 'binary_crossentropy',
        },
        loss_weights={
            'class_output'  : 1.0,
            'decoder_output': BETA / 1024.0,
        },
        metrics={
            'class_output'  : ['accuracy'],
            'decoder_output': ['mse'],
        },
    )
    return model

# ── Baseline transfer model ─────────────────────────────────────────
transfer_model = build_transfer_model('MobileNetV2Transfer_Baseline')
transfer_model.summary()

In [ ]:
transfer_history = train_model(
    transfer_model,
    make_batches_cifar(x_train_tl[subset_idx], y_train[subset_idx], x_train_seg_tl[subset_idx]),
    make_batches_cifar(x_test_tl, y_test, x_test_seg_tl),
    epochs=EPOCHS_BASELINE,
    callbacks=[make_lr_schedule(EPOCHS_BASELINE)],
)

In [ ]:
transfer_acc, transfer_mse = evaluate(transfer_model, x_test_tl, y_test)
print(f"MobileNetV2 Transfer Baseline — Accuracy: {transfer_acc*100:.2f}%  MSE: {transfer_mse:.6f}")
results['transfer'] = {'accuracy': transfer_acc, 'recon_mse': transfer_mse}

In [ ]:
# ── Human-in-the-loop transfer model ───────────────────────────────
transfer_human_model = build_transfer_model('MobileNetV2Transfer_Human')

In [ ]:
transfer_human_history = train_model(
    transfer_human_model,
    make_approved_batches(x_train_tl, y_train, x_train_seg_tl, APPROVED_PATH_TL),
    make_batches_cifar(x_test_tl, y_test, x_test_seg_tl),
    epochs=EPOCHS_HUMAN,
    callbacks=[make_lr_schedule(EPOCHS_HUMAN)],
)

In [ ]:
transfer_human_acc, transfer_human_mse = evaluate(transfer_human_model, x_test_tl, y_test)
print(f"MobileNetV2 Transfer Human — Accuracy: {transfer_human_acc*100:.2f}%  MSE: {transfer_human_mse:.6f}")
results['transfer_human'] = {'accuracy': transfer_human_acc, 'recon_mse': transfer_human_mse}

In [ ]:
if 'transfer' in results and 'transfer_human' in results:
   bt = results['transfer']
   ht = results['transfer_human']
   print('=' * 55)
   print(f'{"":30s}  {"Baseline":>10s}  {"Human":>10s}')
   print('-' * 55)
   print(f'{"Test accuracy":30s}  {bt["accuracy"]*100:>9.2f}%  {ht["accuracy"]*100:>9.2f}%')
   print(f'{"Reconstruction MSE":30s}  {bt["recon_mse"]:>10.6f}  {ht["recon_mse"]:>10.6f}')
   delta_acc = (ht['accuracy'] - bt['accuracy']) * 100
   delta_mse = ht['recon_mse'] - bt['recon_mse']
   direction_acc = 'improvement' if delta_acc > 0 else 'decline'
   direction_mse = 'improvement' if delta_mse < 0 else 'increase'
   print('-' * 55)
   print(f'  Accuracy delta : {delta_acc:+.2f}%  ({direction_acc})')
   print(f'  MSE delta      : {delta_mse:+.6f}  ({direction_mse})')
   print('=' * 55)

   fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 4))
   fig.suptitle('MobileNetV2 Transfer — Baseline vs. Human-in-the-Loop', fontsize=13, fontweight='bold')
   labels = ['Baseline', 'Human']
   colors = ['#4c72b0', '#dd8452']

   accs = [bt['accuracy'] * 100, ht['accuracy'] * 100]
   bars1 = ax1.bar(labels, accs, color=colors, width=0.5)
   ax1.set_ylabel('Test Accuracy (%)')
   ax1.set_title('Classification Accuracy')
   ax1.set_ylim(max(0, min(accs) - 5), min(100, max(accs) + 5))
   for bar, v in zip(bars1, accs):
       ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.2,
                f'{v:.2f}%', ha='center', va='bottom', fontsize=11)

   mses = [bt['recon_mse'], ht['recon_mse']]
   bars2 = ax2.bar(labels, mses, color=colors, width=0.5)
   ax2.set_ylabel('Reconstruction MSE')
   ax2.set_title('Reconstruction MSE  (lower = better)')
   ax2.set_ylim(0, max(mses) * 1.25)
   for bar, v in zip(bars2, mses):
       ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + max(mses) * 0.02,
                f'{v:.5f}', ha='center', va='bottom', fontsize=10)

   plt.tight_layout()
   plt.savefig(os.path.join(OUT_DIR, 'transfer_comparison.png'), dpi=150, bbox_inches='tight')
   plt.show()

print('Transfer Learning — Adversarial Evaluation')

In [ ]:
transfer_fgsm_accs = []
transfer_human_fgsm_accs = []
transfer_pgd_accs = []
transfer_human_pgd_accs = []

In [ ]:
for eps in epsilons:
    # FGSM
    x_adv = fgsm_attack(transfer_model, x_eval, y_eval, eps)
    preds, _ = transfer_model(x_adv, training=False)
    transfer_fgsm_accs.append(np.mean(np.argmax(preds.numpy(), axis=1) == y_eval))

    x_adv = fgsm_attack(transfer_human_model, x_eval, y_eval, eps)
    preds, _ = transfer_human_model(x_adv, training=False)
    transfer_human_fgsm_accs.append(np.mean(np.argmax(preds.numpy(), axis=1) == y_eval))

    # PGD
    x_adv = pgd_attack(transfer_model, x_eval, y_eval, eps)
    preds, _ = transfer_model(x_adv, training=False)
    transfer_pgd_accs.append(np.mean(np.argmax(preds.numpy(), axis=1) == y_eval))

    x_adv = pgd_attack(transfer_human_model, x_eval, y_eval, eps)
    preds, _ = transfer_human_model(x_adv, training=False)
    transfer_human_pgd_accs.append(np.mean(np.argmax(preds.numpy(), axis=1) == y_eval))

print(f"{'Epsilon':<10} {'Base FGSM':>10} {'Human FGSM':>12} {'Delta':>10}")
print("-" * 46)
for eps, b, h in zip(epsilons, transfer_fgsm_accs, transfer_human_fgsm_accs):
    print(f"{eps:<10.2f} {b*100:>9.2f}% {h*100:>11.2f}% {(h-b)*100:>+9.2f}%")

plt.figure(figsize=(10, 5))
plt.plot(epsilons, [a * 100 for a in transfer_fgsm_accs], 'o-', label='MobileNetV2 Baseline (FGSM)')
plt.plot(epsilons, [a * 100 for a in transfer_human_fgsm_accs], 's-', label='MobileNetV2 Human (FGSM)')
plt.plot(epsilons, [a * 100 for a in transfer_pgd_accs], 'o--', label='MobileNetV2 Baseline (PGD)')
plt.plot(epsilons, [a * 100 for a in transfer_human_pgd_accs], 's--', label='MobileNetV2 Human (PGD)')
plt.xlabel('Epsilon (attack strength)')
plt.ylabel('Accuracy (%)')
plt.title('Adversarial Robustness — MobileNetV2 Transfer Learning')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/Adversarial Robustness Research/CIFAR Code/adversarial_transfer_mobilenet.png')
plt.show()

### ResNet50

In [ ]:
from keras.applications import ResNet50
from keras.applications.resnet import preprocess_input as resnet_preprocess

In [ ]:
# ── Preprocess for ResNet50 ────────────────────────────────────────
# ResNet50 expects inputs preprocessed with resnet preprocess_input
# which scales [0,255] -> channel-wise zero-centered (not [-1,1])
def resnet_preprocess_batch(imgs_01):
    """imgs_01: float32 array in [0,1] -> ResNet50-normalized"""
    return resnet_preprocess(imgs_01 * 255.0)

x_train_r50 = resnet_preprocess_batch(x_train)
x_test_r50  = resnet_preprocess_batch(x_test)

In [ ]:
x_train_seg_r50 = resnet_preprocess_batch(x_train_seg)
x_test_seg_r50  = resnet_preprocess_batch(x_test_seg)

In [ ]:
# preprocess approved candidates
raw_approved_r50     = np.load(APPROVED_PATH)
approved_imgs_r50    = resnet_preprocess_batch(raw_approved_r50['imgs'])
approved_labels_r50  = raw_approved_r50['labels']
np.savez('/content/approved_r50.npz',
         imgs=approved_imgs_r50, labels=approved_labels_r50)
APPROVED_PATH_R50 = '/content/approved_r50.npz'

#### ResNet50 — Baseline

In [ ]:
def build_resnet50_model(name='ResNet50Transfer'):
    base = ResNet50(
        weights='imagenet',
        include_top=False,
        input_shape=(32, 32, 3),
    )

    # freeze all layers except the last 35 (same strategy as MobileNetV2)
    for layer in base.layers[:-35]:
        layer.trainable = False

    img_in = Input(shape=(32, 32, 3), name='image_input')
    x = base(img_in, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(0.3)(x)

    # classification head
    class_out = layers.Dense(10, activation='softmax', name='class_output')(x)

    # decoder head — mirrors MobileNetV2 decoder
    gx = layers.Dense(8 * 8 * 64, use_bias=False)(x)
    gx = layers.BatchNormalization()(gx)
    gx = layers.Activation('relu')(gx)
    gx = layers.Reshape((8, 8, 64))(gx)
    gx = layers.Conv2DTranspose(64, (3, 3), strides=(2, 2), padding='same', use_bias=False)(gx)
    gx = layers.BatchNormalization()(gx)
    gx = layers.Activation('relu')(gx)
    gx = layers.Conv2DTranspose(32, (3, 3), strides=(2, 2), padding='same', use_bias=False)(gx)
    gx = layers.BatchNormalization()(gx)
    gx = layers.Activation('relu')(gx)
    recon_out = layers.Conv2D(3, (3, 3), padding='same', activation='sigmoid',
                              name='decoder_output')(gx)

    model = Model(inputs=img_in, outputs=[class_out, recon_out], name=name)
    model.compile(
        optimizer=Adam(LR),
        loss={
            'class_output'  : 'sparse_categorical_crossentropy',
            'decoder_output': 'binary_crossentropy',
        },
        loss_weights={
            'class_output'  : 1.0,
            'decoder_output': BETA / 1024.0,
        },
        metrics={
            'class_output'  : ['accuracy'],
            'decoder_output': ['mse'],
        },
    )
    return model

# ── Baseline ResNet50 ──────────────────────────────────────────────
print('=' * 60)
print('RESNET50 TRANSFER — BASELINE')
print('=' * 60)

In [ ]:
r50_baseline = build_resnet50_model('ResNet50Transfer_Baseline')
r50_baseline.summary()

In [ ]:
r50_baseline_history = train_model(
    r50_baseline,
    make_batches_cifar(x_train_r50[subset_idx], y_train[subset_idx], x_train_seg_r50[subset_idx]),
    make_batches_cifar(x_test_r50, y_test, x_test_seg_r50),
    epochs=EPOCHS_BASELINE,
    callbacks=[make_lr_schedule(EPOCHS_BASELINE)],
)

In [ ]:
r50_baseline_acc, r50_baseline_mse = evaluate(r50_baseline, x_test_r50, y_test)
print(f"ResNet50 Baseline — Accuracy: {r50_baseline_acc*100:.2f}%  MSE: {r50_baseline_mse:.6f}")

#### ResNet50 — Human-in-the-Loop

In [ ]:
print('=' * 60)
print('RESNET50 TRANSFER — HUMAN')
print('=' * 60)

In [ ]:
r50_human = build_resnet50_model('ResNet50Transfer_Human')

In [ ]:
r50_human_history = train_model(
    r50_human,
    make_approved_batches(x_train_r50, y_train, x_train_seg_r50, APPROVED_PATH_R50),
    make_batches_cifar(x_test_r50, y_test, x_test_seg_r50),
    epochs=EPOCHS_HUMAN,
    callbacks=[make_lr_schedule(EPOCHS_HUMAN)],
)

In [ ]:
r50_human_acc, r50_human_mse = evaluate(r50_human, x_test_r50, y_test)
print(f"ResNet50 Human — Accuracy: {r50_human_acc*100:.2f}%  MSE: {r50_human_mse:.6f}")

#### ResNet50 — Comparison

In [ ]:
print('=' * 55)
print(f'{"":30s}  {"Baseline":>10s}  {"Human":>10s}')
print('-' * 55)
print(f'{"Test accuracy":30s}  {r50_baseline_acc*100:>9.2f}%  {r50_human_acc*100:>9.2f}%')
print(f'{"Reconstruction MSE":30s}  {r50_baseline_mse:>10.6f}  {r50_human_mse:>10.6f}')
delta_acc = (r50_human_acc - r50_baseline_acc) * 100
delta_mse = r50_human_mse - r50_baseline_mse
print('-' * 55)
print(f'  Accuracy delta : {delta_acc:+.2f}%')
print(f'  MSE delta      : {delta_mse:+.6f}')
print('=' * 55)

#### ResNet50 — Adversarial Evaluation

In [ ]:
print('ResNet50 Transfer — Adversarial Evaluation')

In [ ]:
# resnet50-aware attack bounds
R50_MIN = x_test_r50.min()
R50_MAX = x_test_r50.max()

In [ ]:
def fgsm_attack_r50(model, images, labels, epsilon):
    images = tf.cast(images, tf.float32)
    with tf.GradientTape() as tape:
        tape.watch(images)
        preds, _ = model(images, training=False)
        loss = keras.losses.sparse_categorical_crossentropy(labels, preds)
    gradients = tape.gradient(loss, images)
    perturbations = epsilon * tf.sign(gradients)
    return tf.clip_by_value(images + perturbations, R50_MIN, R50_MAX)

def pgd_attack_r50(model, images, labels, epsilon, alpha=None, num_steps=40):
    if alpha is None:
        alpha = epsilon / 10.0  # scale step size relative to epsilon
    x_adv = tf.identity(images)
    for _ in range(num_steps):
        with tf.GradientTape() as tape:
            tape.watch(x_adv)
            preds, _ = model(x_adv, training=False)
            loss = keras.losses.sparse_categorical_crossentropy(labels, preds)
        gradients = tape.gradient(loss, x_adv)
        x_adv = x_adv + alpha * tf.sign(gradients)
        x_adv = tf.clip_by_value(x_adv, images - epsilon, images + epsilon)
        x_adv = tf.clip_by_value(x_adv, R50_MIN, R50_MAX)
    return x_adv

r50_baseline_fgsm, r50_human_fgsm = [], []
r50_baseline_pgd,  r50_human_pgd  = [], []
x_eval = x_test_r50[:2000]
y_eval = y_test[:2000]

In [ ]:
epsilons_r50 = [eps * 255.0 for eps in epsilons]

In [ ]:
for eps in epsilons_r50:
    # FGSM
    x_adv = fgsm_attack_r50(r50_baseline, x_eval, y_eval, eps)
    preds, _ = r50_baseline(x_adv, training=False)
    r50_baseline_fgsm.append(np.mean(np.argmax(preds.numpy(), axis=1) == y_eval))

    x_adv = fgsm_attack_r50(r50_human, x_eval, y_eval, eps)
    preds, _ = r50_human(x_adv, training=False)
    r50_human_fgsm.append(np.mean(np.argmax(preds.numpy(), axis=1) == y_eval))

    # PGD
    x_adv = pgd_attack_r50(r50_baseline, x_eval, y_eval, eps)
    preds, _ = r50_baseline(x_adv, training=False)
    r50_baseline_pgd.append(np.mean(np.argmax(preds.numpy(), axis=1) == y_eval))

    x_adv = pgd_attack_r50(r50_human, x_eval, y_eval, eps)
    preds, _ = r50_human(x_adv, training=False)
    r50_human_pgd.append(np.mean(np.argmax(preds.numpy(), axis=1) == y_eval))

print(f"\nFGSM — ResNet50 Transfer")
print(f"{'Epsilon':<10} {'Baseline':>10} {'Human':>10} {'Delta':>10}")
print("-" * 46)
for eps, b, h in zip(epsilons, r50_baseline_fgsm, r50_human_fgsm):
    print(f"{eps:<10.3f} {b*100:>9.2f}% {h*100:>9.2f}% {(h-b)*100:>+9.2f}%")

print(f"\nPGD — ResNet50 Transfer")
print(f"{'Epsilon':<10} {'Baseline':>10} {'Human':>10} {'Delta':>10}")
print("-" * 46)
for eps, b, h in zip(epsilons, r50_baseline_pgd, r50_human_pgd):
    print(f"{eps:<10.3f} {b*100:>9.2f}% {h*100:>9.2f}% {(h-b)*100:>+9.2f}%")

#### ResNet50 — Save Models

In [ ]:
r50_baseline.save('/content/drive/MyDrive/Adversarial Robustness Research/CIFAR Code/r50_baseline.keras')
r50_human.save('/content/drive/MyDrive/Adversarial Robustness Research/CIFAR Code/r50_human.keras')
print("ResNet50 models saved.")

### VGG16 & VGG19
Shared helper functions for the VGG16/VGG19 transfer-learning experiments, used by both variants below.

In [ ]:
from keras.applications import VGG16, VGG19
from keras.applications.vgg16 import preprocess_input as vgg_preprocess

In [ ]:
# vgg16 and vgg19 share the same preprocessing (channel-wise zero-centering)
def vgg_preprocess_batch(imgs_01):
    return vgg_preprocess(imgs_01 * 255.0)

x_train_vgg = vgg_preprocess_batch(x_train)
x_test_vgg  = vgg_preprocess_batch(x_test)

In [ ]:
x_train_seg_vgg = vgg_preprocess_batch(x_train_seg)
x_test_seg_vgg  = vgg_preprocess_batch(x_test_seg)

In [ ]:
# preprocess approved candidates
raw_approved_vgg    = np.load(APPROVED_PATH)
approved_imgs_vgg   = vgg_preprocess_batch(raw_approved_vgg['imgs'])
approved_labels_vgg = raw_approved_vgg['labels']
np.savez('/content/approved_vgg.npz',
         imgs=approved_imgs_vgg, labels=approved_labels_vgg)
APPROVED_PATH_VGG = '/content/approved_vgg.npz'

In [ ]:
# vgg attack bounds
VGG_MIN = x_test_vgg.min()
VGG_MAX = x_test_vgg.max()

In [ ]:
def fgsm_attack_vgg(model, images, labels, epsilon):
    images = tf.cast(images, tf.float32)
    with tf.GradientTape() as tape:
        tape.watch(images)
        preds, _ = model(images, training=False)
        loss = keras.losses.sparse_categorical_crossentropy(labels, preds)
    gradients = tape.gradient(loss, images)
    perturbations = epsilon * tf.sign(gradients)
    return tf.clip_by_value(images + perturbations, VGG_MIN, VGG_MAX)

def pgd_attack_vgg(model, images, labels, epsilon, alpha=None, num_steps=40):
    if alpha is None:
        alpha = epsilon / 10.0
    x_adv = tf.identity(images)
    for _ in range(num_steps):
        with tf.GradientTape() as tape:
            tape.watch(x_adv)
            preds, _ = model(x_adv, training=False)
            loss = keras.losses.sparse_categorical_crossentropy(labels, preds)
        gradients = tape.gradient(loss, x_adv)
        x_adv = x_adv + alpha * tf.sign(gradients)
        x_adv = tf.clip_by_value(x_adv, images - epsilon, images + epsilon)
        x_adv = tf.clip_by_value(x_adv, VGG_MIN, VGG_MAX)
    return x_adv

# shared eval set for vgg
x_eval_vgg = x_test_vgg[:2000]
y_eval_vgg = y_test[:2000]
epsilons_vgg = [eps * 255.0 for eps in epsilons]

In [ ]:
def build_vgg_model(variant='vgg16', name='VGGTransfer'):
    # shared builder for both VGG16 and VGG19 -- only the last 4 layers
    # are unfrozen (VGG has far more layers than MobileNet/ResNet, so a
    # smaller fine-tuning window is used)
    if variant == 'vgg16':
        base = VGG16(weights='imagenet', include_top=False, input_shape=(32, 32, 3))
    else:
        base = VGG19(weights='imagenet', include_top=False, input_shape=(32, 32, 3))

    # freeze all but last 4 layers — vgg has no batch norm so fewer layers to unfreeze
    for layer in base.layers[:-4]:
        layer.trainable = False

    img_in = Input(shape=(32, 32, 3), name='image_input')
    x = base(img_in, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(0.3)(x)

    # classification head
    class_out = layers.Dense(10, activation='softmax', name='class_output')(x)

    # decoder head
    gx = layers.Dense(8 * 8 * 64, use_bias=False)(x)
    gx = layers.BatchNormalization()(gx)
    gx = layers.Activation('relu')(gx)
    gx = layers.Reshape((8, 8, 64))(gx)
    gx = layers.Conv2DTranspose(64, (3, 3), strides=(2, 2), padding='same', use_bias=False)(gx)
    gx = layers.BatchNormalization()(gx)
    gx = layers.Activation('relu')(gx)
    gx = layers.Conv2DTranspose(32, (3, 3), strides=(2, 2), padding='same', use_bias=False)(gx)
    gx = layers.BatchNormalization()(gx)
    gx = layers.Activation('relu')(gx)
    recon_out = layers.Conv2D(3, (3, 3), padding='same', activation='sigmoid',
                              name='decoder_output')(gx)

    model = Model(inputs=img_in, outputs=[class_out, recon_out], name=name)
    model.compile(
        optimizer=Adam(LR),
        loss={
            'class_output'  : 'sparse_categorical_crossentropy',
            'decoder_output': 'binary_crossentropy',
        },
        loss_weights={
            'class_output'  : 1.0,
            'decoder_output': BETA / 1024.0,
        },
        metrics={
            'class_output'  : ['accuracy'],
            'decoder_output': ['mse'],
        },
    )
    return model

In [ ]:
def run_vgg_experiment(variant='vgg16'):
    # trains + evaluates (clean and adversarial) both the baseline and
    # human-in-the-loop version of a VGG16/VGG19 transfer model, and
    # saves both to Drive -- used for both VGG16 and VGG19 below
    label = variant.upper()
    print('=' * 60)
    print(f'{label} TRANSFER — BASELINE')
    print('=' * 60)

    baseline = build_vgg_model(variant, name=f'{label}_Baseline')
    train_model(
        baseline,
        make_batches_cifar(x_train_vgg[subset_idx], y_train[subset_idx], x_train_seg_vgg[subset_idx]),
        make_batches_cifar(x_test_vgg, y_test, x_test_seg_vgg),
        epochs=EPOCHS_BASELINE,
        callbacks=[make_lr_schedule(EPOCHS_BASELINE)],
    )
    baseline_acc, baseline_mse = evaluate(baseline, x_test_vgg, y_test)
    print(f"{label} Baseline — Accuracy: {baseline_acc*100:.2f}%  MSE: {baseline_mse:.6f}")

    print('=' * 60)
    print(f'{label} TRANSFER — HUMAN')
    print('=' * 60)

    human = build_vgg_model(variant, name=f'{label}_Human')
    train_model(
        human,
        make_approved_batches(x_train_vgg, y_train, x_train_seg_vgg, APPROVED_PATH_VGG),
        make_batches_cifar(x_test_vgg, y_test, x_test_seg_vgg),
        epochs=EPOCHS_HUMAN,
        callbacks=[make_lr_schedule(EPOCHS_HUMAN)],
    )
    human_acc, human_mse = evaluate(human, x_test_vgg, y_test)
    print(f"{label} Human — Accuracy: {human_acc*100:.2f}%  MSE: {human_mse:.6f}")

    # summary
    print('=' * 55)
    print(f'{"":30s}  {"Baseline":>10s}  {"Human":>10s}')
    print('-' * 55)
    print(f'{"Test accuracy":30s}  {baseline_acc*100:>9.2f}%  {human_acc*100:>9.2f}%')
    print(f'{"Reconstruction MSE":30s}  {baseline_mse:>10.6f}  {human_mse:>10.6f}')
    delta_acc = (human_acc - baseline_acc) * 100
    delta_mse = human_mse - baseline_mse
    print('-' * 55)
    print(f'  Accuracy delta : {delta_acc:+.2f}%')
    print(f'  MSE delta      : {delta_mse:+.6f}')
    print('=' * 55)

    # adversarial eval
    print(f'{label} Transfer — Adversarial Evaluation')
    baseline_fgsm, human_fgsm = [], []
    baseline_pgd,  human_pgd  = [], []

    for eps in epsilons_vgg:
        x_adv = fgsm_attack_vgg(baseline, x_eval_vgg, y_eval_vgg, eps)
        preds, _ = baseline(x_adv, training=False)
        baseline_fgsm.append(np.mean(np.argmax(preds.numpy(), axis=1) == y_eval_vgg))

        x_adv = fgsm_attack_vgg(human, x_eval_vgg, y_eval_vgg, eps)
        preds, _ = human(x_adv, training=False)
        human_fgsm.append(np.mean(np.argmax(preds.numpy(), axis=1) == y_eval_vgg))

        x_adv = pgd_attack_vgg(baseline, x_eval_vgg, y_eval_vgg, eps, alpha=eps/10.0)
        preds, _ = baseline(x_adv, training=False)
        baseline_pgd.append(np.mean(np.argmax(preds.numpy(), axis=1) == y_eval_vgg))

        x_adv = pgd_attack_vgg(human, x_eval_vgg, y_eval_vgg, eps, alpha=eps/10.0)
        preds, _ = human(x_adv, training=False)
        human_pgd.append(np.mean(np.argmax(preds.numpy(), axis=1) == y_eval_vgg))

    print(f"\nFGSM — {label} Transfer")
    print(f"{'Epsilon':<10} {'Baseline':>10} {'Human':>10} {'Delta':>10}")
    print("-" * 46)
    for eps, b, h in zip(epsilons, baseline_fgsm, human_fgsm):
        print(f"{eps:<10.3f} {b*100:>9.2f}% {h*100:>9.2f}% {(h-b)*100:>+9.2f}%")

    print(f"\nPGD — {label} Transfer")
    print(f"{'Epsilon':<10} {'Baseline':>10} {'Human':>10} {'Delta':>10}")
    print("-" * 46)
    for eps, b, h in zip(epsilons, baseline_pgd, human_pgd):
        print(f"{eps:<10.3f} {b*100:>9.2f}% {h*100:>9.2f}% {(h-b)*100:>+9.2f}%")

    # save models
    baseline.save(f'/content/drive/MyDrive/Adversarial Robustness Research/CIFAR Code/{variant}_baseline.keras')
    human.save(f'/content/drive/MyDrive/Adversarial Robustness Research/CIFAR Code/{variant}_human.keras')
    print(f"{label} models saved.")

    return baseline, human, baseline_acc, human_acc, baseline_mse, human_mse, baseline_fgsm, human_fgsm, baseline_pgd, human_pgd

#### Run VGG16

In [ ]:
vgg16_results = run_vgg_experiment('vgg16')

#### Run VGG19

In [ ]:
vgg19_results = run_vgg_experiment('vgg19')

## 10. Final Results Summary
Prints every clean-accuracy and adversarial-robustness table computed above, gathered in one place for easy copying into a report.

In [ ]:
# ══════════════════════════════════════════════════════════════════
# FINAL RESULTS SUMMARY — CIFAR-10
# ══════════════════════════════════════════════════════════════════

In [ ]:
def section(title):
    print('\n' + '═' * 65)
    print(f'  {title}')
    print('═' * 65)


In [ ]:
def subsection(title):
    print(f'\n── {title} ──')


In [ ]:
def acc_table(headers, rows):
    col_w = 12
    header_str = ''.join(f'{h:>{col_w}}' for h in headers)
    print(header_str)
    print('-' * (col_w * len(headers)))
    for row in rows:
        label = row[0]
        vals  = row[1:]
        print(f'{label:<20}' + ''.join(f'{v:>{col_w}}' for v in vals))


In [ ]:
def rob_table(epsilons, *conditions):
    # conditions: list of (label, accs_list)
    headers = ['ε'] + [c[0] for c in conditions]
    col_w = 12
    print(''.join(f'{h:>{col_w}}' for h in headers))
    print('-' * (col_w * len(headers)))
    for i, eps in enumerate(epsilons):
        row = f'{eps:<12.3f}'
        for _, accs in conditions:
            row += f'{accs[i]*100:>11.2f}%'
        print(row)


In [ ]:
# ── 1. Custom model — clean accuracy & MSE ────────────────────────
section('1. CUSTOM MODEL — CLEAN ACCURACY & RECONSTRUCTION MSE')
acc_table(
    ['', 'Baseline', 'Auto', 'Human'],
    [
  ['Test Accuracy',     f'{baseline_acc*100:.2f}%',       f'{auto_acc*100:.2f}%',  f'{human_acc:.2f}%'],
  ['Reconstruction MSE', f'{baseline_mse:.5f}',           f'{auto_mse:.5f}',       f'{human_mse:.5f}'],
      ]
)
# NOTE: `f'{human_acc:.2f}%'` above is missing the `*100` that the
# Baseline/Auto columns use (`f'{baseline_acc*100:.2f}%'`), so as
# written this prints the Human accuracy as a raw fraction (e.g.
# '0.85%' instead of '85.00%'). Left unchanged since it's a display-only
# formatting bug in a print statement -- doesn't affect any saved
# results -- but worth fixing before you screenshot this table.

In [ ]:
# ── 2. Custom model — FGSM ────────────────────────────────────────
section('2. CUSTOM MODEL — FGSM ADVERSARIAL ROBUSTNESS')
rob_table(epsilons,
    ('Baseline', baseline_fgsm_accs),
    ('Auto',     auto_fgsm_accs),
    ('Human',    human_fgsm_accs),
)

In [ ]:
# ── 3. Custom model — PGD ─────────────────────────────────────────
section('3. CUSTOM MODEL — PGD ADVERSARIAL ROBUSTNESS')
rob_table(epsilons,
    ('Baseline', baseline_pgd_accs),
    ('Auto',     auto_pgd_accs),
    ('Human',    human_pgd_accs),
)

In [ ]:
# ── 4. MobileNetV2 transfer — clean accuracy ──────────────────────
section('4. MOBILENETV2 TRANSFER — CLEAN ACCURACY & MSE')
acc_table(
    ['', 'Baseline', 'Human'],
    [
        ['Test Accuracy',      f'{transfer_acc*100:.2f}%',       f'{transfer_human_acc*100:.2f}%'],
        ['Reconstruction MSE', f'{transfer_mse:.5f}',            f'{transfer_human_mse:.5f}'],
    ]
)

In [ ]:
# ── 5. MobileNetV2 transfer — FGSM ───────────────────────────────
section('5. MOBILENETV2 TRANSFER — FGSM ADVERSARIAL ROBUSTNESS')
rob_table(epsilons,
    ('Baseline', transfer_fgsm_accs),
    ('Human',    transfer_human_fgsm_accs),
)

In [ ]:
# ── 6. MobileNetV2 transfer — PGD ────────────────────────────────
section('6. MOBILENETV2 TRANSFER — PGD ADVERSARIAL ROBUSTNESS')
rob_table(epsilons,
    ('Baseline', transfer_pgd_accs),
    ('Human',    transfer_human_pgd_accs),
)

In [ ]:
# ── 7. ResNet50 transfer — clean accuracy ─────────────────────────
section('7. RESNET50 TRANSFER — CLEAN ACCURACY')
acc_table(
    ['', 'Baseline', 'Human'],
    [
        ['Test Accuracy', f'{r50_baseline_acc*100:.2f}%', f'{r50_human_acc*100:.2f}%'],
    ]
)

In [ ]:
# ── 8. ResNet50 transfer — FGSM ──────────────────────────────────
section('8. RESNET50 TRANSFER — FGSM ADVERSARIAL ROBUSTNESS')
rob_table(epsilons,
    ('Baseline', r50_baseline_fgsm),
    ('Human',    r50_human_fgsm),
)

In [ ]:
# ── 9. ResNet50 transfer — PGD ───────────────────────────────────
section('9. RESNET50 TRANSFER — PGD ADVERSARIAL ROBUSTNESS')
rob_table(epsilons,
    ('Baseline', r50_baseline_pgd),
    ('Human',    r50_human_pgd),
)

In [ ]:
# ── 10. VGG16 transfer — clean accuracy ──────────────────────────
section('10. VGG16 TRANSFER — CLEAN ACCURACY')
vgg16_baseline, vgg16_human, vgg16_baseline_acc, vgg16_human_acc, \
vgg16_baseline_mse, vgg16_human_mse, vgg16_baseline_fgsm, vgg16_human_fgsm, \
vgg16_baseline_pgd, vgg16_human_pgd = vgg16_results

In [ ]:
acc_table(
    ['', 'Baseline', 'Human'],
    [
        ['Test Accuracy', f'{vgg16_baseline_acc*100:.2f}%', f'{vgg16_human_acc*100:.2f}%'],
    ]
)

In [ ]:
# ── 11. VGG16 transfer — FGSM ────────────────────────────────────
section('11. VGG16 TRANSFER — FGSM ADVERSARIAL ROBUSTNESS')
rob_table(epsilons,
    ('Baseline', vgg16_baseline_fgsm),
    ('Human',    vgg16_human_fgsm),
)

In [ ]:
# ── 12. VGG16 transfer — PGD ─────────────────────────────────────
section('12. VGG16 TRANSFER — PGD ADVERSARIAL ROBUSTNESS')
rob_table(epsilons,
    ('Baseline', vgg16_baseline_pgd),
    ('Human',    vgg16_human_pgd),
)

In [ ]:
# ── 13. VGG19 transfer — clean accuracy ──────────────────────────
section('13. VGG19 TRANSFER — CLEAN ACCURACY')
vgg19_baseline, vgg19_human, vgg19_baseline_acc, vgg19_human_acc, \
vgg19_baseline_mse, vgg19_human_mse, vgg19_baseline_fgsm, vgg19_human_fgsm, \
vgg19_baseline_pgd, vgg19_human_pgd = vgg19_results

In [ ]:
acc_table(
    ['', 'Baseline', 'Human'],
    [
        ['Test Accuracy', f'{vgg19_baseline_acc*100:.2f}%', f'{vgg19_human_acc*100:.2f}%'],
    ]
)

In [ ]:
# ── 14. VGG19 transfer — FGSM ────────────────────────────────────
section('14. VGG19 TRANSFER — FGSM ADVERSARIAL ROBUSTNESS')
rob_table(epsilons,
    ('Baseline', vgg19_baseline_fgsm),
    ('Human',    vgg19_human_fgsm),
)

In [ ]:
# ── 15. VGG19 transfer — PGD ─────────────────────────────────────
section('15. VGG19 TRANSFER — PGD ADVERSARIAL ROBUSTNESS')
rob_table(epsilons,
    ('Baseline', vgg19_baseline_pgd),
    ('Human',    vgg19_human_pgd),
)

In [ ]:
print('\n' + '═' * 65)
print('  END OF CIFAR-10 RESULTS')
print('═' * 65)